In [1]:
# ============================================================
# 12.1 SYNTHETIC V2 MODEL OPTIMISATION
# Dataset Discovery and Initial Audit
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd()

# If notebook is executed from notebooks/ folder
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SYNTHETIC_DIR = PROJECT_ROOT / "datasets" / "Synthetic"

print("=" * 70)
print("SYNTHETIC V2 MODEL OPTIMISATION")
print("=" * 70)

print("\nProject root:")
print(PROJECT_ROOT)

print("\nSynthetic directory:")
print(SYNTHETIC_DIR)

print("\nDirectory exists:", SYNTHETIC_DIR.exists())

# ------------------------------------------------------------
# Find CSV files
# ------------------------------------------------------------

csv_files = list(SYNTHETIC_DIR.rglob("*.csv"))

print("\nCSV files found:")
print("-" * 70)

for file in csv_files:
    print(file.relative_to(PROJECT_ROOT))

# ------------------------------------------------------------
# Identify the V2 dataset
# ------------------------------------------------------------

v2_candidates = [
    file for file in csv_files
    if "v2" in file.name.lower()
]

if not v2_candidates:
    raise FileNotFoundError(
        "Synthetic V2 CSV was not found inside datasets/Synthetic/"
    )

print("\nV2 candidate files:")
for file in v2_candidates:
    print(file)

# Use the largest V2 CSV if multiple candidates exist
SYNTHETIC_V2_PATH = max(
    v2_candidates,
    key=lambda x: x.stat().st_size
)

print("\nSelected dataset:")
print(SYNTHETIC_V2_PATH)

# ------------------------------------------------------------
# Load dataset
# ------------------------------------------------------------

synthetic_df = pd.read_csv(SYNTHETIC_V2_PATH)

print("\n" + "=" * 70)
print("DATASET LOADED")
print("=" * 70)

print("Rows   :", synthetic_df.shape[0])
print("Columns:", synthetic_df.shape[1])

display(synthetic_df.head())

print("\nColumn names:")
for i, col in enumerate(synthetic_df.columns, start=1):
    print(f"{i:02d}. {col}")

SYNTHETIC V2 MODEL OPTIMISATION

Project root:
d:\newwwwwwww\AiBasedInstagramPrediction

Synthetic directory:
d:\newwwwwwww\AiBasedInstagramPrediction\datasets\Synthetic

Directory exists: True

CSV files found:
----------------------------------------------------------------------
datasets\Synthetic\synthetic_instagram_engagement_dataset_100k.csv
datasets\Synthetic\synthetic_instagram_engagement_dataset_v2.csv

V2 candidate files:
d:\newwwwwwww\AiBasedInstagramPrediction\datasets\Synthetic\synthetic_instagram_engagement_dataset_v2.csv

Selected dataset:
d:\newwwwwwww\AiBasedInstagramPrediction\datasets\Synthetic\synthetic_instagram_engagement_dataset_v2.csv

DATASET LOADED
Rows   : 100000
Columns: 62


,post_id,account_id,category,account_type,follower_count,following_count,account_age_days,verified_status,posting_frequency,average_historical_engagement,...,brightness,contrast,saturation,sharpness,colorfulness,face_count,text_in_image,visual_complexity,estimated_image_quality,performance_class
0,POST_00000000,ACC_003617,Education,Business,2272,1269,1010,False,7.87,0.027265,...,0.484,0.697,0.423,0.320,0.599,2.0,0.0,0.327,0.650,High
1,POST_00000001,ACC_000534,Technology,Personal,267,372,2767,False,2.60,0.041254,...,0.680,0.775,0.260,0.784,0.612,0.0,0.0,0.122,0.682,Medium
2,POST_00000002,ACC_006323,Entertainment,Creator,7642,1225,1437,False,2.04,0.016957,...,0.423,0.356,0.362,0.563,0.755,1.0,0.0,0.411,0.640,Medium
3,POST_00000003,ACC_009502,Education,Creator,3880,1187,2569,False,8.76,0.017244,...,NaN,0.527,0.447,0.394,0.777,1.0,0.0,0.197,0.601,Low
4,POST_00000004,ACC_002845,Education,Personal,2104,521,305,False,7.29,0.014833,...,0.581,0.602,0.623,0.570,0.263,3.0,0.0,0.505,0.649,Low



Column names:
01. post_id
02. account_id
03. category
04. account_type
05. follower_count
06. following_count
07. account_age_days
08. verified_status
09. posting_frequency
10. average_historical_engagement
11. audience_growth_rate
12. account_activity_level
13. content_consistency
14. caption
15. caption_length
16. word_count
17. sentence_count
18. hashtags
19. hashtag_count
20. unique_hashtag_count
21. average_hashtag_length
22. hashtag_character_count
23. emoji_count
24. mention_count
25. question_mark_count
26. exclamation_count
27. uppercase_ratio
28. numeric_token_count
29. url_present
30. call_to_action
31. caption_sentiment
32. caption_subjectivity
33. caption_readability
34. keyword_density
35. caption_complexity
36. caption_engagement_intent
37. posting_datetime
38. posting_hour
39. day_of_week
40. is_weekend
41. posting_time_period
42. media_type
43. has_image
44. has_location
45. has_mention
46. sponsored
47. content_originality
48. content_quality_score
49. creator_activi

In [2]:
# ============================================================
# 12.2 TARGET AND LEAKAGE AUDIT
# ============================================================

print("=" * 70)
print("TARGET AND LEAKAGE AUDIT")
print("=" * 70)

TARGET = "performance_class"

# ------------------------------------------------------------
# Verify target
# ------------------------------------------------------------

if TARGET not in synthetic_df.columns:
    raise ValueError(
        f"Target column '{TARGET}' was not found."
    )

print("\nTarget column:")
print(TARGET)

print("\nTarget data type:")
print(synthetic_df[TARGET].dtype)

# ------------------------------------------------------------
# Target distribution
# ------------------------------------------------------------

target_distribution = (
    synthetic_df[TARGET]
    .value_counts(dropna=False)
    .rename_axis("Performance_Class")
    .reset_index(name="Count")
)

target_distribution["Percentage"] = (
    target_distribution["Count"]
    / len(synthetic_df)
    * 100
).round(2)

print("\n" + "=" * 70)
print("TARGET DISTRIBUTION")
print("=" * 70)

display(target_distribution)

# ------------------------------------------------------------
# Missing target values
# ------------------------------------------------------------

missing_target = synthetic_df[TARGET].isna().sum()

print("\nMissing target values:", missing_target)

if missing_target > 0:
    raise ValueError(
        "Target contains missing values. Resolve this before modelling."
    )

# ------------------------------------------------------------
# Leakage columns
# ------------------------------------------------------------

POST_PUBLICATION_COLUMNS = [
    "likes",
    "comments",
    "shares",
    "saves",
    "reach",
    "impressions",
    "engagement_rate",
    "binary_performance"
]

IDENTIFIER_COLUMNS = [
    "post_id",
    "account_id",
    "data_source"
]

LEAKAGE_COLUMNS = [
    col for col in POST_PUBLICATION_COLUMNS + IDENTIFIER_COLUMNS
    if col in synthetic_df.columns
]

print("\n" + "=" * 70)
print("LEAKAGE AUDIT")
print("=" * 70)

print("\nPost-publication variables found:")
for col in POST_PUBLICATION_COLUMNS:
    if col in synthetic_df.columns:
        print("  EXCLUDE:", col)

print("\nIdentifier variables found:")
for col in IDENTIFIER_COLUMNS:
    if col in synthetic_df.columns:
        print("  EXCLUDE:", col)

# ------------------------------------------------------------
# Predictor columns
# ------------------------------------------------------------

PREDICTOR_COLUMNS = [
    col
    for col in synthetic_df.columns
    if col not in LEAKAGE_COLUMNS + [TARGET]
]

print("\n" + "=" * 70)
print("PREDICTOR AUDIT")
print("=" * 70)

print("\nTotal columns:", len(synthetic_df.columns))
print("Target:", TARGET)
print("Excluded leakage/identifier columns:", len(LEAKAGE_COLUMNS))
print("Legitimate predictor columns:", len(PREDICTOR_COLUMNS))

print("\nLegitimate predictors:")

for i, col in enumerate(PREDICTOR_COLUMNS, start=1):
    print(f"{i:02d}. {col}")

# ------------------------------------------------------------
# Check for suspicious target-related columns
# ------------------------------------------------------------

suspicious_keywords = [
    "performance",
    "engagement",
    "target",
    "label",
    "outcome"
]

suspicious_columns = []

for col in PREDICTOR_COLUMNS:

    col_lower = col.lower()

    if any(
        keyword in col_lower
        for keyword in suspicious_keywords
    ):
        suspicious_columns.append(col)

print("\n" + "=" * 70)
print("SUSPICIOUS PREDICTOR REVIEW")
print("=" * 70)

if suspicious_columns:

    print(
        "\nColumns containing potentially target-related keywords:"
    )

    for col in suspicious_columns:
        print("  REVIEW:", col)

else:

    print("\nNo suspicious predictor names detected.")

# ------------------------------------------------------------
# Final leakage confirmation
# ------------------------------------------------------------

remaining_forbidden = [
    col
    for col in PREDICTOR_COLUMNS
    if col in POST_PUBLICATION_COLUMNS
    or col in IDENTIFIER_COLUMNS
]

print("\n" + "=" * 70)

if len(remaining_forbidden) == 0:

    print("LEAKAGE AUDIT PASSED")
    print(
        "No prohibited post-publication metrics or identifiers "
        "are included as predictors."
    )

else:

    print("LEAKAGE AUDIT FAILED")
    print("Remaining forbidden columns:")
    print(remaining_forbidden)

print("=" * 70)

TARGET AND LEAKAGE AUDIT

Target column:
performance_class

Target data type:
str

TARGET DISTRIBUTION


,Performance_Class,Count,Percentage
0,Medium,36000,36.0
1,High,32000,32.0
2,Low,32000,32.0



Missing target values: 0

LEAKAGE AUDIT

Post-publication variables found:

Identifier variables found:
  EXCLUDE: post_id
  EXCLUDE: account_id

PREDICTOR AUDIT

Total columns: 62
Target: performance_class
Excluded leakage/identifier columns: 2
Legitimate predictor columns: 59

Legitimate predictors:
01. category
02. account_type
03. follower_count
04. following_count
05. account_age_days
06. verified_status
07. posting_frequency
08. average_historical_engagement
09. audience_growth_rate
10. account_activity_level
11. content_consistency
12. caption
13. caption_length
14. word_count
15. sentence_count
16. hashtags
17. hashtag_count
18. unique_hashtag_count
19. average_hashtag_length
20. hashtag_character_count
21. emoji_count
22. mention_count
23. question_mark_count
24. exclamation_count
25. uppercase_ratio
26. numeric_token_count
27. url_present
28. call_to_action
29. caption_sentiment
30. caption_subjectivity
31. caption_readability
32. keyword_density
33. caption_complexity
34. c

In [3]:
# ============================================================
# 12.3 FEATURE–TARGET SIGNAL ANALYSIS
# ============================================================

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import mutual_info_score
import pandas as pd
import numpy as np

print("=" * 70)
print("FEATURE–TARGET SIGNAL ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# Create encoded target
# ------------------------------------------------------------

target_encoder = LabelEncoder()

y_encoded = target_encoder.fit_transform(
    synthetic_df[TARGET].astype(str)
)

print("\nTarget encoding:")
for class_name, encoded_value in zip(
    target_encoder.classes_,
    target_encoder.transform(target_encoder.classes_)
):
    print(f"  {class_name} -> {encoded_value}")

# ------------------------------------------------------------
# Separate numeric and categorical features
# ------------------------------------------------------------

numeric_features = synthetic_df[PREDICTOR_COLUMNS].select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = synthetic_df[PREDICTOR_COLUMNS].select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("\n" + "=" * 70)
print("FEATURE TYPES")
print("=" * 70)

print("Numeric features    :", len(numeric_features))
print("Categorical features:", len(categorical_features))

# ------------------------------------------------------------
# Numeric feature analysis
# ------------------------------------------------------------

numeric_results = []

for feature in numeric_features:

    series = pd.to_numeric(
        synthetic_df[feature],
        errors="coerce"
    )

    # Median imputation only for this diagnostic
    series = series.fillna(series.median())

    # Skip constant features
    if series.nunique() <= 1:
        continue

    try:

        mi = mutual_info_classif(
            series.to_numpy().reshape(-1, 1),
            y_encoded,
            random_state=42
        )[0]

    except Exception:
        mi = np.nan

    # Class means
    class_means = (
        synthetic_df
        .assign(_feature=series)
        .groupby(TARGET)["_feature"]
        .mean()
    )

    numeric_results.append({
        "Feature": feature,
        "Mutual_Information": mi,
        "Overall_Mean": series.mean(),
        "Low_Mean": class_means.get("Low", np.nan),
        "Medium_Mean": class_means.get("Medium", np.nan),
        "High_Mean": class_means.get("High", np.nan),
        "Missing_Count": synthetic_df[feature].isna().sum()
    })

numeric_signal_df = pd.DataFrame(numeric_results)

numeric_signal_df = numeric_signal_df.sort_values(
    "Mutual_Information",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 70)
print("TOP NUMERIC FEATURES BY MUTUAL INFORMATION")
print("=" * 70)

display(
    numeric_signal_df.head(20)
)

# ------------------------------------------------------------
# Categorical feature analysis
# ------------------------------------------------------------

categorical_results = []

for feature in categorical_features:

    values = synthetic_df[feature].astype(str)

    # Frequency encode rare/missing values
    encoded_values, _ = pd.factorize(values)

    try:

        mi = mutual_info_score(
            encoded_values,
            y_encoded
        )

    except Exception:
        mi = np.nan

    categorical_results.append({
        "Feature": feature,
        "Mutual_Information": mi,
        "Unique_Values": values.nunique(),
        "Missing_Count": synthetic_df[feature].isna().sum()
    })

categorical_signal_df = pd.DataFrame(
    categorical_results
)

categorical_signal_df = categorical_signal_df.sort_values(
    "Mutual_Information",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 70)
print("CATEGORICAL FEATURES BY MUTUAL INFORMATION")
print("=" * 70)

display(
    categorical_signal_df
)

# ------------------------------------------------------------
# Category distribution by target
# ------------------------------------------------------------

if "category" in synthetic_df.columns:

    category_target_table = pd.crosstab(
        synthetic_df["category"],
        synthetic_df[TARGET],
        normalize="index"
    ).round(4)

    print("\n" + "=" * 70)
    print("CATEGORY → PERFORMANCE CLASS DISTRIBUTION")
    print("=" * 70)

    display(category_target_table)

# ------------------------------------------------------------
# Account type distribution by target
# ------------------------------------------------------------

if "account_type" in synthetic_df.columns:

    account_type_target_table = pd.crosstab(
        synthetic_df["account_type"],
        synthetic_df[TARGET],
        normalize="index"
    ).round(4)

    print("\n" + "=" * 70)
    print("ACCOUNT TYPE → PERFORMANCE CLASS DISTRIBUTION")
    print("=" * 70)

    display(account_type_target_table)

# ------------------------------------------------------------
# Save signal analysis
# ------------------------------------------------------------

results_dir = PROJECT_ROOT / "results"
results_dir.mkdir(
    parents=True,
    exist_ok=True
)

numeric_signal_df.to_csv(
    results_dir / "v2_numeric_feature_signal.csv",
    index=False
)

categorical_signal_df.to_csv(
    results_dir / "v2_categorical_feature_signal.csv",
    index=False
)

if "category" in synthetic_df.columns:
    category_target_table.to_csv(
        results_dir / "v2_category_target_distribution.csv"
    )

if "account_type" in synthetic_df.columns:
    account_type_target_table.to_csv(
        results_dir / "v2_account_type_target_distribution.csv"
    )

print("\n" + "=" * 70)
print("FEATURE–TARGET SIGNAL ANALYSIS COMPLETED")
print("=" * 70)

FEATURE–TARGET SIGNAL ANALYSIS

Target encoding:
  High -> 0
  Low -> 1
  Medium -> 2

FEATURE TYPES
Numeric features    : 45
Categorical features: 14

TOP NUMERIC FEATURES BY MUTUAL INFORMATION


,Feature,Mutual_Information,Overall_Mean,Low_Mean,Medium_Mean,High_Mean,Missing_Count
0,word_count,0.295798,13.474930,9.877344,14.108639,16.359594,0
1,caption_engagement_intent,0.278728,0.347471,0.164962,0.365345,0.509871,0
2,caption_readability,0.267761,68.366129,55.185842,71.589095,77.920578,0
3,content_quality_score,0.241557,0.627469,0.533729,0.643251,0.703453,0
4,call_to_action,0.240109,0.615830,0.186188,0.695472,0.955875,0
5,caption_length,0.236723,85.830030,67.699844,89.042333,100.346375,0
6,uppercase_ratio,0.178375,0.030839,0.027850,0.031479,0.033109,0
7,sentence_count,0.152821,2.736640,2.129812,2.829528,3.238969,0
8,caption_subjectivity,0.123731,0.051598,0.010294,0.049256,0.095538,0
9,posting_hour,0.063656,12.521010,11.069469,12.276917,14.247156,0



CATEGORICAL FEATURES BY MUTUAL INFORMATION


,Feature,Mutual_Information,Unique_Values,Missing_Count
0,hashtags,1.096977,99995,0
1,posting_datetime,1.057306,95484,0
2,caption,0.900509,67988,0
3,posting_time_period,0.050306,5,0
4,category,0.005297,5,0
5,media_type,0.003115,4,0
6,day_of_week,0.001778,7,0
7,is_weekend,0.001699,2,0
8,verified_status,0.000531,2,0
9,account_type,0.000165,5,0



CATEGORY → PERFORMANCE CLASS DISTRIBUTION


performance_class,High,Low,Medium
category,,,
Education,0.3048,0.3346,0.3606
Entertainment,0.3906,0.2411,0.3683
Fashion,0.3150,0.3247,0.3603
Food,0.3133,0.3248,0.3619
Technology,0.2764,0.3747,0.3489



ACCOUNT TYPE → PERFORMANCE CLASS DISTRIBUTION


performance_class,High,Low,Medium
account_type,,,
Brand,0.3329,0.3131,0.3540
Business,0.3213,0.3227,0.3560
Creator,0.3196,0.3204,0.3600
Influencer,0.3274,0.3091,0.3636
Personal,0.3098,0.3269,0.3633



FEATURE–TARGET SIGNAL ANALYSIS COMPLETED


In [4]:
# ============================================================
# 12.4 HIGH-SIGNAL FEATURE VALIDATION
# ============================================================

from sklearn.feature_selection import f_classif
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
import pandas as pd
import numpy as np

print("=" * 70)
print("HIGH-SIGNAL FEATURE VALIDATION")
print("=" * 70)

# ------------------------------------------------------------
# Important candidate features
# ------------------------------------------------------------

candidate_features = [
    "word_count",
    "caption_engagement_intent",
    "caption_readability",
    "content_quality_score",
    "call_to_action",
    "caption_length",
    "uppercase_ratio",
    "sentence_count",
    "caption_subjectivity",
    "posting_hour",
    "question_mark_count",
    "caption_sentiment",
    "keyword_density",
    "hashtag_count",
    "unique_hashtag_count",
    "creator_activity_score",
    "content_consistency",
    "average_historical_engagement",
    "follower_count"
]

candidate_features = [
    col for col in candidate_features
    if col in synthetic_df.columns
]

print("\nCandidate features found:")
for col in candidate_features:
    print(" ", col)

# ------------------------------------------------------------
# Prepare numeric matrix
# ------------------------------------------------------------

X_signal = synthetic_df[candidate_features].copy()

for col in X_signal.columns:

    X_signal[col] = pd.to_numeric(
        X_signal[col],
        errors="coerce"
    )

    X_signal[col] = X_signal[col].fillna(
        X_signal[col].median()
    )

# ------------------------------------------------------------
# ANOVA F-score
# ------------------------------------------------------------

f_scores, p_values = f_classif(
    X_signal,
    y_encoded
)

anova_results = pd.DataFrame({
    "Feature": candidate_features,
    "F_Score": f_scores,
    "P_Value": p_values
})

anova_results["Significant"] = (
    anova_results["P_Value"] < 0.001
)

anova_results = anova_results.sort_values(
    "F_Score",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 70)
print("ANOVA FEATURE RANKING")
print("=" * 70)

display(anova_results)

# ------------------------------------------------------------
# Class means and standard deviations
# ------------------------------------------------------------

class_statistics = []

for feature in candidate_features:

    temp = synthetic_df[[feature, TARGET]].copy()

    temp[feature] = pd.to_numeric(
        temp[feature],
        errors="coerce"
    )

    class_stats = (
        temp
        .groupby(TARGET)[feature]
        .agg(["mean", "std", "median"])
    )

    row = {
        "Feature": feature
    }

    for class_name in ["Low", "Medium", "High"]:

        if class_name in class_stats.index:

            row[f"{class_name}_Mean"] = (
                class_stats.loc[class_name, "mean"]
            )

            row[f"{class_name}_Std"] = (
                class_stats.loc[class_name, "std"]
            )

            row[f"{class_name}_Median"] = (
                class_stats.loc[class_name, "median"]
            )

    class_statistics.append(row)

class_statistics_df = pd.DataFrame(
    class_statistics
)

print("\n" + "=" * 70)
print("CLASS-LEVEL FEATURE STATISTICS")
print("=" * 70)

display(class_statistics_df)

# ------------------------------------------------------------
# Check whether features are nearly deterministic
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FEATURE RANGE OVERLAP CHECK")
print("=" * 70)

overlap_results = []

for feature in candidate_features:

    ranges = {}

    for class_name in ["Low", "Medium", "High"]:

        values = pd.to_numeric(
            synthetic_df.loc[
                synthetic_df[TARGET] == class_name,
                feature
            ],
            errors="coerce"
        ).dropna()

        if len(values) > 0:

            ranges[class_name] = (
                values.min(),
                values.max()
            )

    overlap_results.append({
        "Feature": feature,
        "Low_Min": ranges.get("Low", (np.nan, np.nan))[0],
        "Low_Max": ranges.get("Low", (np.nan, np.nan))[1],
        "Medium_Min": ranges.get("Medium", (np.nan, np.nan))[0],
        "Medium_Max": ranges.get("Medium", (np.nan, np.nan))[1],
        "High_Min": ranges.get("High", (np.nan, np.nan))[0],
        "High_Max": ranges.get("High", (np.nan, np.nan))[1]
    })

overlap_df = pd.DataFrame(overlap_results)

display(overlap_df)

# ------------------------------------------------------------
# Correlation between candidate features
# ------------------------------------------------------------

correlation_matrix = X_signal.corr()

print("\n" + "=" * 70)
print("HIGH-SIGNAL FEATURE CORRELATION")
print("=" * 70)

display(correlation_matrix.round(3))

# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

results_dir = PROJECT_ROOT / "results"
results_dir.mkdir(
    parents=True,
    exist_ok=True
)

anova_results.to_csv(
    results_dir / "v2_high_signal_anova.csv",
    index=False
)

class_statistics_df.to_csv(
    results_dir / "v2_high_signal_class_statistics.csv",
    index=False
)

overlap_df.to_csv(
    results_dir / "v2_high_signal_range_overlap.csv",
    index=False
)

correlation_matrix.to_csv(
    results_dir / "v2_high_signal_correlations.csv"
)

print("\n" + "=" * 70)
print("HIGH-SIGNAL FEATURE VALIDATION COMPLETED")
print("=" * 70)

HIGH-SIGNAL FEATURE VALIDATION

Candidate features found:
  word_count
  caption_engagement_intent
  caption_readability
  content_quality_score
  call_to_action
  caption_length
  uppercase_ratio
  sentence_count
  caption_subjectivity
  posting_hour
  question_mark_count
  caption_sentiment
  keyword_density
  hashtag_count
  unique_hashtag_count
  creator_activity_score
  content_consistency
  average_historical_engagement
  follower_count

ANOVA FEATURE RANKING


,Feature,F_Score,P_Value,Significant
0,caption_engagement_intent,36642.193458,0.000000e+00,True
1,call_to_action,35575.847340,0.000000e+00,True
2,word_count,35273.999875,0.000000e+00,True
3,content_quality_score,30889.404424,0.000000e+00,True
4,caption_length,27751.677861,0.000000e+00,True
5,sentence_count,17568.517768,0.000000e+00,True
6,caption_readability,13031.188924,0.000000e+00,True
7,caption_subjectivity,9312.207952,0.000000e+00,True
8,question_mark_count,5775.261471,0.000000e+00,True
9,caption_sentiment,4347.986509,0.000000e+00,True



CLASS-LEVEL FEATURE STATISTICS


,Feature,Low_Mean,Low_Std,Low_Median,Medium_Mean,Medium_Std,Medium_Median,High_Mean,High_Std,High_Median
0,word_count,9.877344,2.953665,9.000000,14.108639,3.452889,14.000000,16.359594,2.940163,16.000000
1,caption_engagement_intent,0.164962,0.147996,0.100000,0.365345,0.186072,0.366700,0.509871,0.145183,0.516700
2,caption_readability,55.185842,24.464564,58.250000,71.589095,16.133593,72.890000,77.920578,13.050454,79.810000
3,content_quality_score,0.533729,0.103975,0.543300,0.643251,0.084440,0.646400,0.703453,0.072045,0.704300
4,call_to_action,0.186188,0.389264,0.000000,0.695472,0.460214,1.000000,0.955875,0.205376,1.000000
5,caption_length,67.699844,17.208405,62.000000,89.042333,19.477274,89.000000,100.346375,16.428168,98.000000
6,uppercase_ratio,0.027850,0.009149,0.026000,0.031479,0.008340,0.030300,0.033109,0.007505,0.031200
7,sentence_count,2.129812,0.739876,2.000000,2.829528,0.796057,3.000000,3.238969,0.730371,3.000000
8,caption_subjectivity,0.010294,0.048092,0.000000,0.049256,0.085210,0.000000,0.095538,0.095283,0.119000
9,posting_hour,11.069469,5.834061,10.000000,12.276917,5.452900,12.000000,14.247156,4.269421,15.000000



FEATURE RANGE OVERLAP CHECK


,Feature,Low_Min,Low_Max,Medium_Min,Medium_Max,High_Min,High_Max
0,word_count,6.000000,2.500000e+01,6.000000,2.600000e+01,7.000000,2.800000e+01
1,caption_engagement_intent,0.000000,8.000000e-01,0.000000,8.000000e-01,0.000000,8.000000e-01
2,caption_readability,0.000000,1.000000e+02,1.850000,1.000000e+02,10.840000,1.000000e+02
3,content_quality_score,0.150000,8.999000e-01,0.306200,9.226000e-01,0.419100,9.400000e-01
4,call_to_action,0.000000,1.000000e+00,0.000000,1.000000e+00,0.000000,1.000000e+00
5,caption_length,40.000000,1.420000e+02,40.000000,1.560000e+02,49.000000,1.570000e+02
6,uppercase_ratio,0.016700,8.890000e-02,0.016700,9.430000e-02,0.017900,9.430000e-02
7,sentence_count,1.000000,5.000000e+00,1.000000,5.000000e+00,1.000000,5.000000e+00
8,caption_subjectivity,0.000000,5.000000e-01,0.000000,5.000000e-01,0.000000,5.000000e-01
9,posting_hour,0.000000,2.300000e+01,0.000000,2.300000e+01,0.000000,2.300000e+01



HIGH-SIGNAL FEATURE CORRELATION


,word_count,caption_engagement_intent,caption_readability,content_quality_score,call_to_action,caption_length,uppercase_ratio,sentence_count,caption_subjectivity,posting_hour,question_mark_count,caption_sentiment,keyword_density,hashtag_count,unique_hashtag_count,creator_activity_score,content_consistency,average_historical_engagement,follower_count
word_count,1.000,0.596,0.539,0.633,0.696,0.935,0.278,0.716,0.258,0.025,0.136,0.386,-0.159,-0.001,-0.004,-0.003,-0.001,0.002,0.004
caption_engagement_intent,0.596,1.000,0.287,0.664,0.827,0.635,0.270,0.601,0.625,0.004,0.589,0.332,-0.082,-0.000,-0.001,-0.001,-0.001,0.004,-0.003
caption_readability,0.539,0.287,1.000,0.784,0.295,0.296,0.311,0.305,0.171,0.035,0.157,0.119,-0.132,-0.002,-0.006,-0.008,-0.004,-0.000,0.006
content_quality_score,0.633,0.664,0.784,1.000,0.586,0.494,0.333,0.484,0.411,0.024,0.383,0.237,-0.124,0.053,0.050,-0.005,-0.002,0.002,-0.001
call_to_action,0.696,0.827,0.295,0.586,1.000,0.707,0.235,0.570,0.299,0.003,0.245,0.199,-0.051,0.001,0.001,-0.002,-0.000,0.002,0.003
caption_length,0.935,0.635,0.296,0.494,0.707,1.000,0.202,0.779,0.295,0.014,0.169,0.388,-0.146,-0.000,-0.003,-0.001,0.000,0.002,0.002
uppercase_ratio,0.278,0.270,0.311,0.333,0.235,0.202,1.000,0.418,0.212,0.025,0.273,-0.046,0.009,-0.000,-0.004,-0.001,-0.003,0.003,0.005
sentence_count,0.716,0.601,0.305,0.484,0.570,0.779,0.418,1.000,0.307,0.029,0.285,0.192,-0.103,0.000,-0.004,0.002,0.000,0.002,0.002
caption_subjectivity,0.258,0.625,0.171,0.411,0.299,0.295,0.212,0.307,1.000,0.007,0.804,0.445,-0.080,0.002,0.000,0.002,-0.002,0.004,-0.005
posting_hour,0.025,0.004,0.035,0.024,0.003,0.014,0.025,0.029,0.007,1.000,0.012,-0.003,0.003,0.002,-0.001,0.001,-0.003,0.003,-0.001



HIGH-SIGNAL FEATURE VALIDATION COMPLETED


In [6]:
# ============================================================
# 12.5 SYNTHETIC V2 - ROBUST BASELINE MODELS
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

from sklearn.ensemble import (
    ExtraTreesClassifier,
    RandomForestClassifier,
    HistGradientBoostingClassifier
)

from sklearn.linear_model import LogisticRegression

import pandas as pd
import numpy as np
import time
import warnings

warnings.filterwarnings("ignore")

print("=" * 70)
print("SYNTHETIC V2 - ROBUST BASELINE MODELS")
print("=" * 70)

# ============================================================
# 1. TARGET
# ============================================================

TARGET = "performance_class"

# ============================================================
# 2. EXCLUSIONS
# ============================================================

EXCLUDED_COLUMNS = [
    "post_id",
    "account_id",

    # Post-publication variables
    "likes",
    "comments",
    "shares",
    "saves",
    "reach",
    "impressions",
    "engagement_rate",
    "binary_performance",

    # High-cardinality raw fields
    "caption",
    "hashtags",
    "posting_datetime"
]

EXCLUDED_COLUMNS = [
    col for col in EXCLUDED_COLUMNS
    if col in synthetic_df.columns
]

# ============================================================
# 3. CREATE X AND y
# ============================================================

X = synthetic_df.drop(
    columns=EXCLUDED_COLUMNS + [TARGET],
    errors="ignore"
).copy()

y = synthetic_df[TARGET].astype(str).copy()

print("\nInitial predictors:", X.shape[1])

# ============================================================
# 4. FORCE CORRECT DATA TYPES
# ============================================================

# Convert boolean columns to integers
for col in X.columns:

    if X[col].dtype == "bool":

        X[col] = X[col].astype(int)

# Convert category dtype to string
for col in X.columns:

    if str(X[col].dtype) == "category":

        X[col] = X[col].astype(str)

# ============================================================
# 5. IDENTIFY NUMERIC / CATEGORICAL FEATURES
# ============================================================

numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("\nNumeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

print("\nCategorical columns:")
for col in categorical_features:
    print(
        f"  {col}: "
        f"{X[col].nunique(dropna=False)} unique"
    )

# ============================================================
# 6. REMOVE VERY HIGH-CARDINALITY CATEGORICAL FEATURES
# ============================================================

HIGH_CARDINALITY_LIMIT = 50

high_cardinality = [
    col
    for col in categorical_features
    if X[col].nunique(dropna=False) > HIGH_CARDINALITY_LIMIT
]

if high_cardinality:

    print("\nRemoving high-cardinality categorical features:")

    for col in high_cardinality:
        print(
            f"  {col} -> "
            f"{X[col].nunique(dropna=False):,} unique"
        )

    X = X.drop(
        columns=high_cardinality
    )

    categorical_features = [
        col
        for col in categorical_features
        if col not in high_cardinality
    ]

print("\nFinal predictors:", X.shape[1])

# ============================================================
# 7. FINAL TYPE SAFETY CHECK
# ============================================================

# Numeric columns MUST actually be numeric
for col in numeric_features:

    if col in X.columns:

        X[col] = pd.to_numeric(
            X[col],
            errors="coerce"
        )

# Categorical columns MUST be strings
for col in categorical_features:

    if col in X.columns:

        X[col] = X[col].astype("string")

# Recalculate lists after conversion
numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

print("\nFinal numeric features:", len(numeric_features))
print("Final categorical features:", len(categorical_features))

# ============================================================
# 8. SAFETY ASSERTION
# ============================================================

for col in numeric_features:

    if not pd.api.types.is_numeric_dtype(X[col]):

        raise TypeError(
            f"ERROR: {col} is not numeric."
        )

print("\nData-type validation PASSED.")

# ============================================================
# 9. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("\nTraining records:", len(X_train))
print("Testing records :", len(X_test))

# ============================================================
# 10. PREPROCESSING
# ============================================================

numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median"
        )
    )
])

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="most_frequent"
        )
    ),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ],
    remainder="drop"
)

# ============================================================
# 11. MODELS
# ============================================================

models = {

    "HistGradientBoosting": HistGradientBoostingClassifier(
        learning_rate=0.08,
        max_iter=300,
        max_leaf_nodes=63,
        min_samples_leaf=20,
        l2_regularization=0.1,
        random_state=42
    ),

    "Extra Trees": ExtraTreesClassifier(
        n_estimators=400,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features="sqrt",
        n_jobs=-1,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=400,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features="sqrt",
        n_jobs=-1,
        random_state=42
    ),

    "Logistic Regression": LogisticRegression(
        C=1.0,
        max_iter=3000,
        random_state=42
    )
}

# ============================================================
# 12. TRAIN MODELS
# ============================================================

results = []
trained_models = {}

for name, model in models.items():

    print("\n" + "=" * 70)
    print("TRAINING:", name)
    print("=" * 70)

    start = time.time()

    pipeline = Pipeline([
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            model
        )
    ])

    try:

        pipeline.fit(
            X_train,
            y_train
        )

        predictions = pipeline.predict(
            X_test
        )

        accuracy = accuracy_score(
            y_test,
            predictions
        )

        weighted_f1 = f1_score(
            y_test,
            predictions,
            average="weighted"
        )

        elapsed = time.time() - start

        trained_models[name] = pipeline

        results.append({
            "Model": name,
            "Accuracy": accuracy,
            "Weighted_F1": weighted_f1,
            "Training_Time": elapsed
        })

        print(
            f"Accuracy    : {accuracy:.4%}"
        )

        print(
            f"Weighted F1 : {weighted_f1:.4%}"
        )

        print(
            f"Training time: {elapsed:.2f}s"
        )

    except Exception as e:

        print(
            f"\nFAILED: {name}"
        )

        print(
            type(e).__name__,
            ":",
            str(e)
        )

# ============================================================
# 13. RESULTS
# ============================================================

baseline_results = pd.DataFrame(
    results
)

if len(baseline_results) > 0:

    baseline_results = baseline_results.sort_values(
        "Accuracy",
        ascending=False
    ).reset_index(drop=True)

    print("\n" + "=" * 70)
    print("SYNTHETIC V2 BASELINE RESULTS")
    print("=" * 70)

    display(baseline_results)

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    results_dir = PROJECT_ROOT / "results"
    results_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    baseline_results.to_csv(
        results_dir /
        "v2_structured_baseline_results.csv",
        index=False
    )

    best_accuracy = baseline_results.iloc[0]["Accuracy"]

    print("\n" + "=" * 70)

    if best_accuracy >= 0.90:

        print("🔥 90%+ ACCURACY ACHIEVED")

    else:

        print("90% TARGET NOT YET ACHIEVED")

    print(
        f"Best accuracy: {best_accuracy:.4%}"
    )

    print(
        f"Gap to 90%: {max(0, 0.90 - best_accuracy):.4%}"
    )

    print("=" * 70)

else:

    print("\nNo model completed successfully.")

SYNTHETIC V2 - ROBUST BASELINE MODELS

Initial predictors: 56

Numeric features: 51
Categorical features: 5

Categorical columns:
  category: 5 unique
  account_type: 5 unique
  day_of_week: 7 unique
  posting_time_period: 5 unique
  media_type: 4 unique

Final predictors: 56

Final numeric features: 51
Final categorical features: 5

Data-type validation PASSED.

Training records: 80000
Testing records : 20000

TRAINING: HistGradientBoosting
Accuracy    : 90.9650%
Weighted F1 : 90.9996%
Training time: 17.12s

TRAINING: Extra Trees
Accuracy    : 83.2700%
Weighted F1 : 83.2597%
Training time: 11.81s

TRAINING: Random Forest
Accuracy    : 82.2000%
Weighted F1 : 82.2241%
Training time: 12.81s

TRAINING: Logistic Regression
Accuracy    : 53.1600%
Weighted F1 : 52.8828%
Training time: 61.68s

SYNTHETIC V2 BASELINE RESULTS


,Model,Accuracy,Weighted_F1,Training_Time
0,HistGradientBoosting,0.90965,0.909996,17.120522
1,Extra Trees,0.83270,0.832597,11.805530
2,Random Forest,0.82200,0.822241,12.807725
3,Logistic Regression,0.53160,0.528828,61.684089



🔥 90%+ ACCURACY ACHIEVED
Best accuracy: 90.9650%
Gap to 90%: 0.0000%


In [7]:
# ============================================================
# 12.6 FIVE-FOLD CROSS-VALIDATION
# Validate the 90%+ Synthetic V2 result
# ============================================================

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
import pandas as pd
import numpy as np
import time

print("=" * 70)
print("SYNTHETIC V2 - 5-FOLD CROSS-VALIDATION")
print("=" * 70)

# ------------------------------------------------------------
# Best model from Cell 5
# ------------------------------------------------------------

cv_model = HistGradientBoostingClassifier(
    learning_rate=0.08,
    max_iter=300,
    max_leaf_nodes=63,
    min_samples_leaf=20,
    l2_regularization=0.1,
    random_state=42
)

cv_pipeline = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "classifier",
        cv_model
    )
])

# ------------------------------------------------------------
# Stratified 5-fold CV
# ------------------------------------------------------------

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("\nDataset size:", len(X))
print("Number of folds:", cv.n_splits)

print("\nStarting cross-validation...")
print("This may take a few minutes.")

start_time = time.time()

cv_results = cross_validate(
    cv_pipeline,
    X,
    y,
    cv=cv,
    scoring={
        "accuracy": "accuracy",
        "weighted_f1": "f1_weighted"
    },
    n_jobs=1,
    return_train_score=True
)

elapsed = time.time() - start_time

# ------------------------------------------------------------
# Fold results
# ------------------------------------------------------------

fold_results = pd.DataFrame({
    "Fold": range(1, 6),
    "Training_Accuracy": cv_results["train_accuracy"],
    "Validation_Accuracy": cv_results["test_accuracy"],
    "Training_Weighted_F1": cv_results["train_weighted_f1"],
    "Validation_Weighted_F1": cv_results["test_weighted_f1"],
    "Fit_Time": cv_results["fit_time"]
})

print("\n" + "=" * 70)
print("FOLD-BY-FOLD RESULTS")
print("=" * 70)

display(
    fold_results.round(4)
)

# ------------------------------------------------------------
# Summary statistics
# ------------------------------------------------------------

mean_accuracy = cv_results["test_accuracy"].mean()
std_accuracy = cv_results["test_accuracy"].std()

mean_f1 = cv_results["test_weighted_f1"].mean()
std_f1 = cv_results["test_weighted_f1"].std()

mean_train_accuracy = cv_results["train_accuracy"].mean()

print("\n" + "=" * 70)
print("CROSS-VALIDATION SUMMARY")
print("=" * 70)

print(
    f"\nMean Validation Accuracy : "
    f"{mean_accuracy:.4%}"
)

print(
    f"Accuracy Std. Deviation  : "
    f"{std_accuracy:.4%}"
)

print(
    f"Mean Validation F1       : "
    f"{mean_f1:.4%}"
)

print(
    f"F1 Std. Deviation        : "
    f"{std_f1:.4%}"
)

print(
    f"Mean Training Accuracy   : "
    f"{mean_train_accuracy:.4%}"
)

print(
    f"Total CV Time            : "
    f"{elapsed:.2f} seconds"
)

# ------------------------------------------------------------
# Generalisation gap
# ------------------------------------------------------------

generalisation_gap = (
    mean_train_accuracy -
    mean_accuracy
)

print(
    f"\nGeneralisation Gap       : "
    f"{generalisation_gap:.4%}"
)

# ------------------------------------------------------------
# 90% validation status
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("90% VALIDATION STATUS")
print("=" * 70)

if mean_accuracy >= 0.90:

    print("🔥 90%+ CROSS-VALIDATED ACCURACY ACHIEVED")

    print(
        f"Mean CV Accuracy: "
        f"{mean_accuracy:.4%}"
    )

    print(
        "\nThe 90%+ performance is consistent "
        "across the five validation folds."
    )

else:

    print("90%+ NOT MAINTAINED DURING CROSS-VALIDATION")

    print(
        f"Mean CV Accuracy: "
        f"{mean_accuracy:.4%}"
    )

    print(
        f"Gap to 90%: "
        f"{max(0, 0.90 - mean_accuracy):.4%}"
    )

print("=" * 70)

# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------

results_dir = PROJECT_ROOT / "results"
results_dir.mkdir(
    parents=True,
    exist_ok=True
)

fold_results.to_csv(
    results_dir /
    "v2_5fold_cross_validation_results.csv",
    index=False
)

summary = pd.DataFrame({
    "Metric": [
        "Mean CV Accuracy",
        "Accuracy Std",
        "Mean CV Weighted F1",
        "F1 Std",
        "Mean Training Accuracy",
        "Generalisation Gap"
    ],
    "Value": [
        mean_accuracy,
        std_accuracy,
        mean_f1,
        std_f1,
        mean_train_accuracy,
        generalisation_gap
    ]
})

summary.to_csv(
    results_dir /
    "v2_cross_validation_summary.csv",
    index=False
)

print("\nSaved:")
print(
    results_dir /
    "v2_5fold_cross_validation_results.csv"
)

print(
    results_dir /
    "v2_cross_validation_summary.csv"
)

SYNTHETIC V2 - 5-FOLD CROSS-VALIDATION

Dataset size: 100000
Number of folds: 5

Starting cross-validation...
This may take a few minutes.

FOLD-BY-FOLD RESULTS


,Fold,Training_Accuracy,Validation_Accuracy,Training_Weighted_F1,Validation_Weighted_F1,Fit_Time
0,1,0.9888,0.9055,0.9888,0.9058,14.1588
1,2,0.9888,0.9109,0.9889,0.9112,14.9848
2,3,0.9878,0.9078,0.9878,0.9081,15.0593
3,4,0.9892,0.9100,0.9892,0.9103,16.2427
4,5,0.9889,0.9062,0.9889,0.9066,15.3874



CROSS-VALIDATION SUMMARY

Mean Validation Accuracy : 90.8070%
Accuracy Std. Deviation  : 0.2077%
Mean Validation F1       : 90.8403%
F1 Std. Deviation        : 0.2065%
Mean Training Accuracy   : 98.8713%
Total CV Time            : 93.07 seconds

Generalisation Gap       : 8.0643%

90% VALIDATION STATUS
🔥 90%+ CROSS-VALIDATED ACCURACY ACHIEVED
Mean CV Accuracy: 90.8070%

The 90%+ performance is consistent across the five validation folds.

Saved:
d:\newwwwwwww\AiBasedInstagramPrediction\results\v2_5fold_cross_validation_results.csv
d:\newwwwwwww\AiBasedInstagramPrediction\results\v2_cross_validation_summary.csv


In [8]:
# ============================================================
# 12.7 HISTGRADIENTBOOSTING HYPERPARAMETER OPTIMIZATION
# ============================================================

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import pandas as pd
import numpy as np
import time

print("=" * 70)
print("HISTGRADIENTBOOSTING HYPERPARAMETER OPTIMIZATION")
print("=" * 70)

# ------------------------------------------------------------
# IMPORTANT:
# X_train / y_train are used for tuning.
# X_test / y_test remain untouched for final evaluation.
# ------------------------------------------------------------

print("\nTraining records:", len(X_train))
print("Final hold-out records:", len(X_test))

# ------------------------------------------------------------
# Base model
# ------------------------------------------------------------

hgb_model = HistGradientBoostingClassifier(
    random_state=42
)

tuning_pipeline = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "classifier",
        hgb_model
    )
])

# ------------------------------------------------------------
# Hyperparameter search space
# ------------------------------------------------------------

param_distributions = {

    "classifier__learning_rate": [
        0.02,
        0.03,
        0.05,
        0.07,
        0.08,
        0.10,
        0.12,
        0.15
    ],

    "classifier__max_iter": [
        150,
        200,
        250,
        300,
        400,
        500
    ],

    "classifier__max_leaf_nodes": [
        15,
        31,
        47,
        63,
        95,
        127
    ],

    "classifier__min_samples_leaf": [
        5,
        10,
        15,
        20,
        30,
        40,
        60
    ],

    "classifier__l2_regularization": [
        0.0,
        0.01,
        0.05,
        0.1,
        0.2,
        0.5,
        1.0
    ]
}

# ------------------------------------------------------------
# 3-fold CV for tuning
# ------------------------------------------------------------

tuning_cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

# ------------------------------------------------------------
# Randomized search
# ------------------------------------------------------------

search = RandomizedSearchCV(
    estimator=tuning_pipeline,
    param_distributions=param_distributions,
    n_iter=25,
    scoring="accuracy",
    cv=tuning_cv,
    random_state=42,
    n_jobs=1,
    verbose=2,
    return_train_score=True
)

print("\nSearch configuration:")
print("  Random configurations : 25")
print("  Cross-validation folds: 3")
print("  Scoring               : Accuracy")
print("\nStarting optimization...")
print("This may take several minutes.")

start_time = time.time()

search.fit(
    X_train,
    y_train
)

elapsed = time.time() - start_time

# ------------------------------------------------------------
# Best parameters
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BEST HYPERPARAMETERS")
print("=" * 70)

print("\nBest CV Accuracy:")
print(
    f"{search.best_score_:.4%}"
)

print("\nBest parameters:")

for parameter, value in search.best_params_.items():
    print(
        f"  {parameter}: {value}"
    )

print(
    f"\nOptimization time: "
    f"{elapsed:.2f} seconds"
)

# ------------------------------------------------------------
# Search results
# ------------------------------------------------------------

search_results = pd.DataFrame(
    search.cv_results_
)

search_results = search_results.sort_values(
    "rank_test_score"
).reset_index(drop=True)

selected_columns = [
    "rank_test_score",
    "mean_test_score",
    "std_test_score",
    "mean_train_score",
    "params"
]

print("\n" + "=" * 70)
print("TOP 10 HYPERPARAMETER CONFIGURATIONS")
print("=" * 70)

display(
    search_results[
        selected_columns
    ].head(10)
)

# ------------------------------------------------------------
# Evaluate BEST tuned model on untouched test set
# ------------------------------------------------------------

best_model = search.best_estimator_

test_predictions = best_model.predict(
    X_test
)

final_test_accuracy = accuracy_score(
    y_test,
    test_predictions
)

final_test_f1 = f1_score(
    y_test,
    test_predictions,
    average="weighted"
)

print("\n" + "=" * 70)
print("TUNED MODEL - FINAL HOLD-OUT TEST")
print("=" * 70)

print(
    f"\nTest Accuracy    : "
    f"{final_test_accuracy:.4%}"
)

print(
    f"Test Weighted F1 : "
    f"{final_test_f1:.4%}"
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        test_predictions
    )
)

# ------------------------------------------------------------
# Compare with previous champion
# ------------------------------------------------------------

previous_accuracy = 0.90965

improvement = (
    final_test_accuracy -
    previous_accuracy
)

print("\n" + "=" * 70)
print("MODEL IMPROVEMENT")
print("=" * 70)

print(
    f"\nPrevious Accuracy : "
    f"{previous_accuracy:.4%}"
)

print(
    f"Tuned Accuracy    : "
    f"{final_test_accuracy:.4%}"
)

print(
    f"Improvement       : "
    f"{improvement:+.4%}"
)

# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------

results_dir = PROJECT_ROOT / "results"
results_dir.mkdir(
    parents=True,
    exist_ok=True
)

search_results.to_csv(
    results_dir /
    "v2_histgradientboosting_tuning_results.csv",
    index=False
)

best_params_df = pd.DataFrame({
    "Parameter": list(search.best_params_.keys()),
    "Value": list(search.best_params_.values())
})

best_params_df.to_csv(
    results_dir /
    "v2_best_histgradientboosting_parameters.csv",
    index=False
)

final_model_results = pd.DataFrame({
    "Model": [
        "Previous HistGradientBoosting",
        "Tuned HistGradientBoosting"
    ],
    "Accuracy": [
        previous_accuracy,
        final_test_accuracy
    ],
    "Weighted_F1": [
        np.nan,
        final_test_f1
    ]
})

final_model_results.to_csv(
    results_dir /
    "v2_tuned_model_comparison.csv",
    index=False
)

print("\nSaved optimization results.")

# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print("\n" + "=" * 70)

if final_test_accuracy >= 0.90:

    print("🔥 FINAL TUNED MODEL REMAINS ABOVE 90%")

else:

    print("TUNED MODEL BELOW 90%")

print(
    f"Final hold-out accuracy: "
    f"{final_test_accuracy:.4%}"
)

print("=" * 70)

HISTGRADIENTBOOSTING HYPERPARAMETER OPTIMIZATION

Training records: 80000
Final hold-out records: 20000

Search configuration:
  Random configurations : 25
  Cross-validation folds: 3
  Scoring               : Accuracy

Starting optimization...
This may take several minutes.
Fitting 3 folds for each of 25 candidates, totalling 75 fits
[CV] END classifier__l2_regularization=0.1, classifier__learning_rate=0.08, classifier__max_iter=500, classifier__max_leaf_nodes=15, classifier__min_samples_leaf=30; total time=   8.9s
[CV] END classifier__l2_regularization=0.1, classifier__learning_rate=0.08, classifier__max_iter=500, classifier__max_leaf_nodes=15, classifier__min_samples_leaf=30; total time=   7.0s
[CV] END classifier__l2_regularization=0.1, classifier__learning_rate=0.08, classifier__max_iter=500, classifier__max_leaf_nodes=15, classifier__min_samples_leaf=30; total time=   8.4s
[CV] END classifier__l2_regularization=0.0, classifier__learning_rate=0.07, classifier__max_iter=250, classi

,rank_test_score,mean_test_score,std_test_score,mean_train_score,params
0,1,0.903838,0.001202,0.986381,"{'classifier__min_samples_leaf': 40, 'classifi..."
1,2,0.903563,0.001018,0.964119,"{'classifier__min_samples_leaf': 15, 'classifi..."
2,3,0.903063,0.001684,0.989862,"{'classifier__min_samples_leaf': 60, 'classifi..."
3,4,0.902938,0.001138,0.981837,"{'classifier__min_samples_leaf': 60, 'classifi..."
4,5,0.902750,0.001782,0.988631,"{'classifier__min_samples_leaf': 5, 'classifie..."
5,6,0.902525,0.001577,0.990075,"{'classifier__min_samples_leaf': 30, 'classifi..."
6,7,0.902338,0.001511,0.989937,"{'classifier__min_samples_leaf': 10, 'classifi..."
7,8,0.902175,0.001551,0.961275,"{'classifier__min_samples_leaf': 30, 'classifi..."
8,9,0.902063,0.001678,0.984056,"{'classifier__min_samples_leaf': 60, 'classifi..."
9,10,0.901900,0.001333,0.977887,"{'classifier__min_samples_leaf': 15, 'classifi..."



TUNED MODEL - FINAL HOLD-OUT TEST

Test Accuracy    : 90.9250%
Test Weighted F1 : 90.9626%

Classification Report:
              precision    recall  f1-score   support

        High       0.94      0.92      0.93      6400
         Low       0.94      0.91      0.93      6400
      Medium       0.86      0.89      0.88      7200

    accuracy                           0.91     20000
   macro avg       0.91      0.91      0.91     20000
weighted avg       0.91      0.91      0.91     20000


MODEL IMPROVEMENT

Previous Accuracy : 90.9650%
Tuned Accuracy    : 90.9250%
Improvement       : -0.0400%

Saved optimization results.

🔥 FINAL TUNED MODEL REMAINS ABOVE 90%
Final hold-out accuracy: 90.9250%
